# Preprocessing for the MIT-BIH arrhythmia dataset

## Initialisation

In [1]:
# Import and mount google drive, if running locally comment out
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies not supported by google colab, if running locally comment out
!pip install wfdb
!pip install pywavelets

# Import packages
import os
from os.path import join as osj
import pandas as pd
import numpy as np
import torch
import wfdb
import pywt
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

# File directories
PROJECT_DIRECTORY = '/content/drive/MyDrive/RAT-Net_ws/'
DATASET_DIRECTORY = PROJECT_DIRECTORY + 'mit-bih-arrhythmia-database-1.0.0/'
SAVE_DIRECTORY = PROJECT_DIRECTORY + "data_splits/"

# Labels for non beats
NON_BEAT_ANNOTATIONS = [
    '[',  # Start of ventricular flutter/fibrillation
    '!',  # Ventricular flutter wave
    ']',  # End of ventricular flutter/fibrillation
    'x',  # Non-conducted P-wave (blocked APC)
    '(',  # Waveform onset
    ')',  # Waveform end
    'p',  # Peak of P-wave
    't',  # Peak of T-wave
    'u',  # Peak of U-wave
    '`',  # PQ junction
    "'",  # J-point
    '^',  # Non-captured pacemaker artifact
    '|',  # Isolated QRS-like artifact
    '~',  # Change in signal quality
    '+',  # Rhythm change
    's',  # ST segment change
    'T',  # T-wave change
    '*',  # Systole
    'D',  # Diastole
    '=',  # Measurement annotation
    '"',  # Comment annotation
    '@'   # Link to external data
]

# Label map designed to map MIT-BIH labels into the AAMI classes
AAMI_CLASSES = {
    # N class
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0, 'B': 0,
    # S class
    'A': 1, 'a': 1, 'J': 1, 'S': 1,
    # V class
    'V': 2, 'E': 2, 'r': 2,
    # F class
    'F': 3,
    # Q Class
    'f': 4, '/': 4, 'Q': 4, '?': 4
}

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 92.4 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 26.9 MB/s eta 0:00:00


## Pre-processing functions

In [2]:
def swt_denoise(signal_pair):
    """
    Denoises a pair of ECG signals using SWT with soft thresholding.

    Parameters:
        signal_pair (Tuple[np.ndarray, np.ndarray]): Pair of 1D signals.

    Returns:
        Tuple[np.ndarray, np.ndarray]: Denoised pair.
    """
    def _denoise(signal):
        # Decompose
        max_level = pywt.swt_max_level(len(signal))
        wavelet = "bior3.1"
        coeffs = pywt.swt(signal, wavelet, level=max_level)


        detail_coeffs = np.array(coeffs[0][1])
        sigma = np.median(np.abs(detail_coeffs)) / 0.6745
        threshold = sigma * np.sqrt(2 * np.log(len(signal)))

        # Threshold coefficients
        new_coeffs = []
        for approx, detail in coeffs:
            detail = pywt.threshold(detail, threshold, mode='soft')
            new_coeffs.append((approx, detail))

        # Reconstruct and return
        return pywt.iswt(new_coeffs, wavelet)

    lead1, lead2 = signal_pair[:, 0], signal_pair[:, 1]
    return _denoise(lead1), _denoise(lead2)

def standardise(signal_pair):
    """
    Standardises a pair of ECG signals.

    Parameters:
        signal_pair (Tuple[np.ndarray, np.ndarray]): Pair of 1D signals.

    Returns:
        Tuple[np.ndarray, np.ndarray]: Standardised signals.
    """
    def _standardise(signal):
        mean_val = np.mean(signal)
        std_val = np.std(signal)
        return (signal - mean_val) / std_val

    lead1, lead2 = signal_pair
    return _standardise(lead1), _standardise(lead2)

def segmentation_fixed_window(signal_pair, annotation, before=100, after=200):
    """
    Segments beats using a fixed window around each R-peak for both leads.

    Parameters:
        signal_pair (Tuple[np.ndarray, np.ndarray]): Tuple of 1D ECG signals (lead1, lead2).
        annotation (wfdb.Annotation): Annotation object with R-peaks.
        before (int): Number of samples before the R-peak.
        after (int): Number of samples after the R-peak.

    Returns:
        segments (List[np.ndarray]): List of 2x300 segments (2 leads).
        labels (List[str]): Corresponding beat labels.
    """
    lead1, lead2 = signal_pair
    r_peaks = annotation.sample
    r_labels = annotation.symbol

    segments = []
    labels = []

    for peak, label in zip(r_peaks, r_labels):
        start = peak - before
        end = peak + after

        # Ensure segment is within signal bounds
        if start < 0 or end > len(lead1):
            continue

        if label not in NON_BEAT_ANNOTATIONS:
            seg1 = lead1[start:end]
            seg2 = lead2[start:end]
            segment = np.stack([seg1, seg2], axis=0) # Shape: [2, window_length]
            segments.append(segment)
            labels.append(AAMI_CLASSES[label])

    return segments, labels

def process_signal(signal_pair, annotation):
    """
    Takes an id and returns the X, y for that id.

    Parameters:
        id (int): The id of the record.

    Returns:
        tuple: A tuple containing the segments and labels.
    """
    denoised_pair = swt_denoise(signal_pair)
    normalised_pair = standardise(denoised_pair)
    segmented_pair, y = segmentation_fixed_window(normalised_pair, annotation)

    X = np.array(segmented_pair)  # Convert to numpy array

    X = torch.tensor(X, dtype=torch.float32)

    y = torch.tensor(y, dtype=torch.long)

    return X, y

## Load, denoise, standardise and segment data. Then split into train test and validation

In [4]:
# Get an array of all ids
record_ids = pd.read_csv(osj(DATASET_DIRECTORY, 'RECORDS'), header=None).to_numpy().reshape(-1)

X = []
y = []

# Load the data
for id in record_ids:
    signal, info = wfdb.io.rdsamp(osj(DATASET_DIRECTORY, str(id)))
    annotation = wfdb.rdann(DATASET_DIRECTORY+ str(id), "atr")
    X_i, y_i = process_signal(signal, annotation)
    X.append(X_i)
    y.append(y_i)
    print(f'Loaded record: {id}')

X = np.concatenate(X, axis=0)
y = np.concatenate(y, axis=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train)

Loaded record: 100
Loaded record: 101
Loaded record: 102
Loaded record: 103
Loaded record: 104
Loaded record: 105
Loaded record: 106
Loaded record: 107
Loaded record: 108
Loaded record: 109
Loaded record: 111
Loaded record: 112
Loaded record: 113
Loaded record: 114
Loaded record: 115
Loaded record: 116
Loaded record: 117
Loaded record: 118
Loaded record: 119
Loaded record: 121
Loaded record: 122
Loaded record: 123
Loaded record: 124
Loaded record: 200
Loaded record: 201
Loaded record: 202
Loaded record: 203
Loaded record: 205
Loaded record: 207
Loaded record: 208
Loaded record: 209
Loaded record: 210
Loaded record: 212
Loaded record: 213
Loaded record: 214
Loaded record: 215
Loaded record: 217
Loaded record: 219
Loaded record: 220
Loaded record: 221
Loaded record: 222
Loaded record: 223
Loaded record: 228
Loaded record: 230
Loaded record: 231
Loaded record: 232
Loaded record: 233
Loaded record: 234


## Apply SMOTE-Tomek to data

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)

# Apply SMOTE-Tomek to the flattened data
smote_tomek = SMOTETomek(sampling_strategy='auto', smote=SMOTE(k_neighbors=5))
X_train_resampled_flat, y_train_resampled = smote_tomek.fit_resample(X_train_flat, y_train)

# Reshape back to the original shape if necessary
X_train_resampled = X_train_resampled_flat.reshape(X_train_resampled_flat.shape[0], X_train.shape[1], X_train.shape[2])

## Save the resampled train data and the test and validation data

In [ ]:
# Make directory if it doesn't exist
os.makedirs(SAVE_DIRECTORY, exist_ok=True)

# Save data splits
try:
  np.save(osj(SAVE_DIRECTORY + "y_train.npy"), y_train_resampled)
  np.save(osj(SAVE_DIRECTORY + "X_train.npy"), X_train_resampled)
  np.save(osj(SAVE_DIRECTORY + "X_val.npy"), X_val)
  np.save(osj(SAVE_DIRECTORY + "y_val.npy"), y_val)
  np.save(osj(SAVE_DIRECTORY + "X_test.npy"), X_test)
  np.save(osj(SAVE_DIRECTORY + "y_test.npy"), y_test)
except:
  print("Error saving data splits")